# 01 — Binance TradFi 资金费率套利回测

**策略**：做 Delta 中性  
- Binance 永续合约方向跟随费率符号（正费率→空永续；负费率→多永续）  
- IBKR 合成期权对冲 Delta（卖Call+买Put 或 买Call+卖Put，ATM，最近月到期）

**标的**：11 只 Binance EQUITY 类永续合约（SPY/QQQ/TSLA/NVDA/META/GOOGL/AMZN/TSM/MSTR/COIN/AAPL）  
**周期**：2026-01 上线至今（约 81 天，每8小时一个信号周期）

**成本模型**：
| 项目 | 金额（名义本金） |
|---|---|
| Binance 期货手续费 | 0.10% / 开仓 |
| IBKR 期权买卖价差（平静期） | 0.023% / 开仓 |
| IBKR 期权买卖价差（高波动期） | 0.072% / 开仓 |
| 期权月度换期摊销 | 0.026% / 月 |
| 借贷成本 | 0.5% / 年 |

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.data.binance import load_all, EQUITY_SYMBOLS
from src.backtest.engine import run, sensitivity
from src.backtest.costs import CostModel, DEFAULT
from src.backtest.metrics import summary, print_summary

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print('依赖加载完成')

## 1. 加载历史费率数据

In [ ]:
# 首次运行会从 Binance API 拉取数据并缓存到 data/*.parquet
wide = load_all()
print(f'数据范围: {wide.index[0]} → {wide.index[-1]}')
print(f'周期数: {len(wide)}  (约 {len(wide)/3:.0f} 天)')
print(f'合约数: {len(wide.columns)}')
print(f'\n各合约行数:')
print((wide != 0).sum().to_string())

In [ ]:
# 年化费率统计（按合约）
ann = wide * 3 * 365 * 100
stats = pd.DataFrame({
    '均值_%/年': ann.mean().round(1),
    '标准差_%/年': ann.std().round(1),
    '最大_%/年': ann.max().round(1),
    '最小_%/年': ann.min().round(1),
    '正费率占比%': (wide > 0).mean().mul(100).round(1),
    '负费率占比%': (wide < 0).mean().mul(100).round(1),
}).rename(index=lambda x: x.replace('USDT', ''))
stats.sort_values('均值_%/年', ascending=False)

## 2. 费率热力图

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
plot_data = (wide * 3 * 365 * 100).rename(columns=lambda x: x.replace('USDT', ''))
# 按时间日期分组取均值（每天3个8h周期 → 日均值）
daily = plot_data.resample('1D').mean()
im = ax.imshow(daily.T, aspect='auto', cmap='RdYlGn', vmin=-300, vmax=300)
ax.set_yticks(range(len(daily.columns)))
ax.set_yticklabels(daily.columns, fontsize=8)
# x轴日期
x_dates = daily.index
step = max(1, len(x_dates) // 10)
ax.set_xticks(range(0, len(x_dates), step))
ax.set_xticklabels([str(d)[:10] for d in x_dates[::step]], rotation=30, ha='right')
plt.colorbar(im, ax=ax, label='年化费率 %/年')
ax.set_title('Binance TradFi 永续合约 — 日均年化费率热力图')
plt.tight_layout()
plt.show()

## 3. 基准回测（默认参数）

In [ ]:
# 基准：阈值=0.01%/8h，5x杠杆，12% IBKR PM 保证金
equity, trades = run(wide, cost=DEFAULT, threshold=0.0001, max_leverage=5.0)
print_summary(equity, label='基准策略')
print(f'\n开仓总次数: {len(trades)}')
if len(trades):
    print(f'平均名义敞口: {trades["notional"].mean():.3f}x 净值')
    print(f'总开仓成本: {trades["open_cost"].sum():.4f} (名义本金)')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)

# ── 净值曲线
axes[0].plot(equity.index, equity.values, color='steelblue', lw=1.5, label='策略净值')
axes[0].axhline(1, color='gray', ls='--', lw=0.8)
axes[0].set_ylabel('净值')
axes[0].set_title('Binance TradFi 资金费率套利 — 净值曲线')
axes[0].legend()

# ── 回撤
dd = equity / equity.cummax() - 1
axes[1].fill_between(dd.index, dd.values, 0, color='red', alpha=0.4, label='回撤')
axes[1].set_ylabel('回撤')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
axes[1].legend()

# ── 活跃合约数
if len(trades):
    active = trades.groupby('ts').size()
    axes[2].bar(active.index, active.values, width=pd.Timedelta('7h'), color='teal', alpha=0.7)
axes[2].set_ylabel('当期开仓合约数')
axes[2].set_xlabel('日期')

plt.tight_layout()
plt.show()

## 4. 开仓分析

In [ ]:
if len(trades):
    tl = trades.copy()
    tl['symbol_short'] = tl['symbol'].str.replace('USDT', '')
    
    # 每个合约开仓统计
    sym_stats = tl.groupby('symbol_short').agg(
        开仓次数=('ts', 'count'),
        平均年化费率=('rate_ann', 'mean'),
        总开仓成本=('open_cost', 'sum'),
        高波动占比=('is_stress', 'mean'),
    ).round(3)
    sym_stats['高波动占比'] = sym_stats['高波动占比'].map('{:.0%}'.format)
    sym_stats.sort_values('开仓次数', ascending=False)

In [ ]:
if len(trades):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    
    # 各合约开仓次数
    cnt = tl.groupby('symbol_short').size().sort_values(ascending=True)
    axes[0].barh(cnt.index, cnt.values, color='steelblue')
    axes[0].set_title('各合约开仓次数')
    axes[0].set_xlabel('次数')
    
    # 各合约平均年化费率
    rate = tl.groupby('symbol_short')['rate_ann'].mean().sort_values(ascending=True)
    colors = ['green' if v > 0 else 'red' for v in rate.values]
    axes[1].barh(rate.index, rate.values, color=colors)
    axes[1].set_title('各合约平均年化费率 %/年')
    axes[1].set_xlabel('%/年')
    axes[1].axvline(0, color='black', lw=0.8)
    
    plt.tight_layout()
    plt.show()

## 5. 敏感性分析（阈值 × 杠杆）

In [ ]:
sens = sensitivity(
    wide,
    thresholds=[0.00003, 0.00005, 0.0001, 0.00015, 0.0002],
    leverages=[3.0, 4.0, 5.0],
    cost=DEFAULT,
)
sens['threshold_label'] = sens['threshold_%/8h'].map(lambda x: f'{x:.4f}%')
print(sens.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (lev, grp) in zip(axes, sens.groupby('leverage')):
    x = range(len(grp))
    ax.bar(x, grp['ann_ret_%'], color='steelblue', alpha=0.8, label='年化收益%')
    ax2 = ax.twinx()
    ax2.plot(x, grp['sharpe'], 'ro-', label='夏普')
    ax2.set_ylabel('夏普比率', color='red')
    ax.set_xticks(list(x))
    ax.set_xticklabels(grp['threshold_label'], rotation=30)
    ax.set_title(f'{lev:.0f}x 杠杆')
    ax.set_ylabel('年化收益 %')
    ax.set_xlabel('入场阈值')
    ax.legend(loc='upper left')
    ax2.legend(loc='upper right')
plt.suptitle('阈值 × 杠杆 敏感性分析', y=1.02)
plt.tight_layout()
plt.show()

## 6. 压力测试：关税冲击期（2026-04-02 后）

In [ ]:
cutoff = '2026-04-02'
wide_normal = wide[wide.index < cutoff]
wide_stress = wide[wide.index >= cutoff]

results = {}
for label, subset in [('全样本', wide), ('正常期', wide_normal), ('关税冲击期', wide_stress)]:
    if len(subset) < 3:
        print(f'{label}: 数据不足，跳过')
        continue
    eq, _ = run(subset, cost=DEFAULT, threshold=0.0001, max_leverage=5.0)
    results[label] = eq
    print_summary(eq, label=label)

# 分段净值对比
fig, ax = plt.subplots(figsize=(12, 4))
colors = ['steelblue', 'green', 'red']
for (label, eq), color in zip(results.items(), colors):
    ax.plot(eq.index, eq.values, label=label, color=color, lw=1.5)
ax.axvline(pd.Timestamp(cutoff, tz='UTC'), color='orange', ls='--', lw=1.2, label='关税冲击开始')
ax.set_title('正常期 vs 关税冲击期净值对比')
ax.legend()
plt.tight_layout()
plt.show()